In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Check if CUDA is available
import torch
if torch.cuda.is_available():
    print(f"CUDA is available! Using GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    print("CUDA is not available, using CPU")
    device = torch.device('cpu')
print(f"Device: {device}")

CUDA is available! Using GPU: NVIDIA H200 NVL
Device: cuda


In [3]:
# Let's explore the repository structure
repo_path = '/net/scratch2/smallyan/rome_eval'
print("Repository structure:")
for root, dirs, files in os.walk(repo_path):
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

Repository structure:
rome_eval/
  globals.yml
  CodeWalkthrough.md
  .gitignore
  plan.md
  CITATION.cff
  documentation.pdf
  LICENSE
  .gitattributes
  util/
    globals.py
    __init__.py
    hparams.py
    runningstats.py
    nethook.py
    generate.py
    perplexity.py
    logit_lens.py
    __pycache__/
      globals.cpython-311.pyc
      perplexity.cpython-311.pyc
      hparams.cpython-311.pyc
      __init__.cpython-311.pyc
      logit_lens.cpython-311.pyc
      runningstats.cpython-311.pyc
      generate.cpython-311.pyc
      nethook.cpython-311.pyc
  hparams/
    FT/
      EleutherAI_gpt-j-6B_unconstr.json
      EleutherAI_gpt-j-6B_constr.json
      gpt2-xl_unconstr.json
      gpt2-medium_constr.json
      gpt2-xl_attn.json
      gpt2-xl_constr.json
      gpt2-large_constr.json
    KE/
      gpt2-xl_zsRE.json
      gpt2-xl_CF.json
      gpt2-xl.json
    MEND/
      gpt2-xl_zsRE.json
      EleutherAI_gpt-j-6B_CF.json
      gpt2-xl.json
      EleutherAI_gpt-j-6B.json
      gpt2-

# Consistency Evaluation for ROME Repository

This notebook evaluates the consistency of the research project at `/net/scratch2/smallyan/rome_eval`.

## Overview
We will perform a binary checklist evaluation:
- **CS1**: Conclusion vs Original Results
- **CS2**: Implementation Follows the Plan

Let's start by reading the key documentation files.

In [4]:
# Read the plan.md file
plan_path = os.path.join(repo_path, 'plan.md')
with open(plan_path, 'r') as f:
    plan_content = f.read()
print("=" * 80)
print("PLAN.MD CONTENT")
print("=" * 80)
print(plan_content)

PLAN.MD CONTENT
# Plan
## Objective
Analyze the storage and recall of factual associations in autoregressive transformer language models, finding evidence that these associations correspond to localized, directly-editable computations.

## Hypothesis
1. Factual associations in GPT correspond to a localized computation mechanism where each midlayer MLP module accepts inputs encoding a subject, then produces outputs recalling memorized properties about that subject, with middle layer MLP outputs accumulating information that is copied to the last token by attention at high layers.
2. Factual associations are localized in the MLP modules at specific middle layers, specifically at the processing of the subject's last token.
3. MLP layers in transformers can be modeled as linear associative memory where weights act as key-value stores.

## Methodology
1. Develop a causal intervention method (Causal Tracing) using causal mediation analysis to identify neuron activations that are decisive in 